<a href="https://colab.research.google.com/github/sdavanipur/Continuous-Thought-Machines/blob/main/Maze_Solving_CTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Continuous Thought Machines (CTMs) on Maze Solving

In this notebook, I sought to implement and train a recently proposed architecture of Deep Neural Networks whose functioning resembles that of their biological counterparts in terms of time, that is to say, their information processing takes place across a number of internal ticks where the neurons' states are updated iteratively, hence, the time coordinate has been added to their functioning in order to make their behavior richer.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML



In [2]:
# Function to generate Synthetic Maze Datasets

class SyntheticMazeDataset(Dataset):
    """
    Generates simple grid mazes with walls, start points, and goal points.
    Input tensor (3 channels): [Channel 0: Walls (1=wall), Channel 1: Start (1), Channel 2: Goal (1)]
    Target tensor (1 channel): [1 where valid shortest path exists, 0 elsewhere]
    """
    def __init__(self, num_samples=1000, grid_size=9, wall_density=0.20, max_attempts=None):
        super().__init__()
        self.num_samples = num_samples
        self.grid_size = grid_size

        if isinstance(wall_density, (tuple, list)) and len(wall_density) == 2:
            self.min_wall_density, self.max_wall_density = wall_density
        else:
            self.min_wall_density = self.max_wall_density = wall_density

        self.data = []

        if max_attempts is None:
            attempts_limit = num_samples * 10
        else:
            attempts_limit = max_attempts

        count = 0
        attempts = 0
        while count < num_samples and attempts < attempts_limit:
            input_grid, path_mask, goal_y, goal_x = self._generate_single_maze() # Now returns goal_y, goal_x
            # Check solvability: goal cell should be marked in path_mask if a path exists
            # path_mask is (H, W) here, not [1, H, W] yet
            if path_mask[goal_y, goal_x] == 1.0:
                self.data.append((torch.tensor(input_grid), torch.tensor(path_mask).unsqueeze(0)))
                count += 1
            attempts += 1
        if count < num_samples:
            print(f"Warning: Could not generate {num_samples} solvable mazes within {attempts_limit} attempts. Generated {count} mazes.")

    def _generate_single_maze(self):
        H = W = self.grid_size
        walls = np.zeros((H, W), dtype=np.float32)

        # Add random wall obstacles
        current_wall_density = np.random.uniform(self.min_wall_density, self.max_wall_density)
        mask = np.random.rand(H, W) < current_wall_density
        walls[mask] = 1.0

        # Identify all clear (non-wall) coordinates
        clear_coords = [(r, c) for r in range(H) for c in range(W) if walls[r, c] == 0.0]

        if len(clear_coords) < 2:
            # Fallback for very dense mazes or small grids where not enough clear spots exist
            start_y, start_x = 0, 0
            goal_y, goal_x = H - 1, W - 1
            # Ensure fallback spots are clear
            walls[start_y, start_x] = 0.0
            walls[goal_y, goal_x] = 0.0
        else:
            max_dist = -1
            best_pairs_candidates = [] # Store (dist, (coord1, coord2))

            # Find all pairs with maximum Manhattan distance
            for i in range(len(clear_coords)): # Iterate through half to avoid redundant pairs
                for j in range(i + 1, len(clear_coords)): # Ensure distinct and avoid duplicate pairs
                    (cy1, cx1) = clear_coords[i]
                    (cy2, cx2) = clear_coords[j]

                    dist = abs(cy1 - cy2) + abs(cx1 - cx2)

                    if dist > max_dist:
                        max_dist = dist
                        best_pairs_candidates = [((cy1, cx1), (cy2, cx2))]
                    elif dist == max_dist:
                        best_pairs_candidates.append(((cy1, cx1), (cy2, cx2)))

            # Randomly select one of the best pairs
            chosen_pair = best_pairs_candidates[np.random.randint(len(best_pairs_candidates))]

            # Randomly assign which is start and which is goal from the chosen pair
            if np.random.rand() < 0.5:
                start_y, start_x = chosen_pair[0]
                goal_y, goal_x = chosen_pair[1]
            else:
                start_y, start_x = chosen_pair[1]
                goal_y, goal_x = chosen_pair[0]

        # Ensure start and goal are clear after potential modifications
        walls[start_y, start_x] = 0.0
        walls[goal_y, goal_x] = 0.0

        # Simple BFS ground truth path solver
        path_mask = np.zeros((H, W), dtype=np.float32)
        queue = [(start_y, start_x, [(start_y, start_x)])]
        visited = set([(start_y, start_x)])
        found_path = None # Initialize to None to indicate no path found yet

        while queue:
            cy, cx, current_path = queue.pop(0)
            if (cy, cx) == (goal_y, goal_x):
                found_path = current_path
                break
            for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ny, nx = cy + dy, cx + dx
                if 0 <= ny < H and 0 <= nx < W and walls[ny, nx] == 0.0:
                    if (ny, nx) not in visited:
                        visited.add((ny, nx))
                        queue.append((ny, nx, current_path + [(ny, nx)]))

        if found_path:
            for py, px in found_path:
                path_mask[py, px] = 1.0

        start_channel = np.zeros((H, W), dtype=np.float32)
        goal_channel = np.zeros((H, W), dtype=np.float32)
        start_channel[start_y, start_x] = 1.0
        goal_channel[goal_y, goal_x] = 1.0

        input_grid = np.stack([walls, start_channel, goal_channel], axis=0) # [3, H, W]
        return input_grid, path_mask, goal_y, goal_x # Return numpy arrays and goal coords

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx]

In [3]:
# Function to visualize CTM reasoning on maze solving

def visualize_maze_ctm_reasoning(model, dataset, sample_idx, device):
    model.eval()
    test_maze, ground_truth = dataset[sample_idx]
    with torch.no_grad():
        _, logits_ticks, sync_mats = model(test_maze.unsqueeze(0).to(device))

    # Convert tick outputs to sigmoid spatial probabilities
    prob_grids = [torch.sigmoid(t_leg).squeeze().cpu().numpy() for t_leg in logits_ticks]
    sync_matrices = [S.squeeze().cpu().numpy() for S in sync_mats]

    walls = test_maze[0].numpy()
    start_channel = test_maze[1].numpy()
    goal_channel = test_maze[2].numpy()

    grid_size = dataset.grid_size
    total_ticks = model.total_ticks

    # Extract dynamic start and goal coordinates
    start_coords = np.argwhere(start_channel == 1)[0] # (y, x)
    goal_coords = np.argwhere(goal_channel == 1)[0]   # (y, x)

    # --- Setup Colab Interactive Animation ---
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Maze CTM Reasoning Trajectory Across Thought Ticks", fontsize=14, fontweight='bold')

    # Subplot 1: Input Maze
    axes[0].imshow(walls, cmap='binary')
    axes[0].plot(start_coords[1], start_coords[0], 'go', markersize=10, label='Start') # plot(x, y)
    axes[0].plot(goal_coords[1], goal_coords[0], 'ro', markersize=10, label='Goal')     # plot(x, y)
    axes[0].set_title("Input Maze & Targets")
    axes[0].legend(loc='lower left')
    axes[0].axis('off')

    # Subplot 2: Predicted Path Probability Heatmap
    im_path = axes[1].imshow(prob_grids[0], cmap='viridis', vmin=0.0, vmax=1.0)
    axes[1].set_title("Path Confidence (Tick t=1)")
    axes[1].axis('off')
    fig.colorbar(im_path, ax=axes[1], fraction=0.046, pad=0.04)

    # Add text for path confidence value
    initial_max_prob = np.max(prob_grids[0])
    path_confidence_text = axes[1].text(0.05, 0.95, f'Max Prob: {initial_max_prob:.3f}',
                                        transform=axes[1].transAxes, color='white',
                                        fontsize=10, ha='left', va='top',
                                        bbox=dict(boxstyle='round,pad=0.3', fc='black', ec='none', alpha=0.6))

    # Subplot 3: Phase Synchrony Matrix S_ij
    im_sync = axes[2].imshow(sync_matrices[0], cmap='magma', vmin=-1.0, vmax=1.0)
    axes[2].set_title("Neuron Synchrony S_ij (Tick t=1)")
    axes[2].axis('off')
    fig.colorbar(im_sync, ax=axes[2], fraction=0.046, pad=0.04)

    def update(t):
        im_path.set_data(prob_grids[t])
        axes[1].set_title(f"Path Confidence (Tick t={t+1}/{total_ticks})")

        # Update path confidence value
        current_max_prob = np.max(prob_grids[t])
        path_confidence_text.set_text(f'Max Prob: {current_max_prob:.3f}')

        im_sync.set_data(sync_matrices[t])
        axes[2].set_title(f"Neuron Synchrony S_ij (Tick t={t+1}/{total_ticks})")
        return im_path, path_confidence_text, im_sync

    anim = FuncAnimation(fig, update, frames=total_ticks, interval=400, blit=False)
    plt.close(fig)

    return HTML(anim.to_html5_video())

In [4]:
# ==========================================
#    Maze-Solving CTM Architecture
# ==========================================



class NeuronLevelModels(nn.Module):
    def __init__(self, num_neurons: int, history_len: int):
        super().__init__()
        self.nlm_conv = nn.Conv1d(
            in_channels=num_neurons,
            out_channels=num_neurons,
            kernel_size=history_len,
            groups=num_neurons,
            bias=True
        )

    def forward(self, pre_history: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.nlm_conv(pre_history).squeeze(-1))



class SynchronyEngine(nn.Module):
    def __init__(self, num_neurons: int, sync_window: int):
        super().__init__()
        self.sync_window = sync_window
        triu_idx = torch.triu_indices(num_neurons, num_neurons, offset=1)
        self.register_buffer("row", triu_idx[0])
        self.register_buffer("col", triu_idx[1])

    def forward(self, post_window: torch.Tensor):
        z_norm = F.normalize(post_window, p=2, dim=-1, eps=1e-8)
        S = torch.bmm(z_norm, z_norm.transpose(1, 2))
        s_vec = S[:, self.row, self.col]
        return S, s_vec



class MazeCTM(nn.Module):
    def __init__(self, grid_size=9, in_channels=3, num_neurons=32, history_len=4, sync_window=4, total_ticks=8, dropout_rate=0.2):
        super().__init__()
        self.grid_size = grid_size
        self.num_neurons = num_neurons
        self.history_len = history_len
        self.sync_window = sync_window
        self.total_ticks = total_ticks

        input_dim = in_channels * grid_size * grid_size
        self.W_syn = nn.Linear(input_dim + num_neurons, num_neurons)
        self.dropout_syn = nn.Dropout(dropout_rate) # Added Dropout
        self.nlms = NeuronLevelModels(num_neurons, history_len)
        self.sync_engine = SynchronyEngine(num_neurons, sync_window)

        sync_dim = (num_neurons * (num_neurons - 1)) // 2
        # Project synchrony features back into spatial maze grid logits [H, W]
        self.W_spatial_out = nn.Sequential(
            nn.Linear(sync_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # Added Dropout
            nn.Linear(128, grid_size * grid_size)
        )

    def forward(self, x: torch.Tensor):
        batch_size = x.size(0)
        device = x.device
        x_flat = x.view(batch_size, -1)

        z_prev = torch.zeros(batch_size, self.num_neurons, device=device)
        pre_hist = torch.zeros(batch_size, self.num_neurons, self.history_len, device=device)
        post_win = torch.zeros(batch_size, self.num_neurons, self.sync_window, device=device)

        logits_per_tick = []
        sync_mats_per_tick = []

        for t in range(self.total_ticks):
            syn_in = torch.cat([x_flat, z_prev], dim=-1)
            a_t = self.dropout_syn(self.W_syn(syn_in)) # Applied dropout
            pre_hist = torch.cat([pre_hist[:, :, 1:], a_t.unsqueeze(-1)], dim=-1)

            z_t = self.nlms(pre_hist)
            z_prev = z_t
            post_win = torch.cat([post_win[:, :, 1:], z_t.unsqueeze(-1)], dim=-1)

            S_t, s_vec_t = self.sync_engine(post_win)
            logits_flat = self.W_spatial_out(s_vec_t)
            logits_grid = logits_flat.view(batch_size, 1, self.grid_size, self.grid_size)

            logits_per_tick.append(logits_grid)
            sync_mats_per_tick.append(S_t)

        return logits_per_tick[-1], logits_per_tick, sync_mats_per_tick



In [5]:
# ==========================================
#    Training Function
# ==========================================

def train_maze_ctm_model(grid_size, total_ticks, wall_density_range, num_neurons, history_len, sync_window, num_samples=500, num_epochs=100, lr=2e-3, dropout_rate=0.2):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Generating synthetic mazes (grid_size={grid_size}, wall_density_range={wall_density_range})... Device: {device}")
    dataset = SyntheticMazeDataset(num_samples=num_samples, grid_size=grid_size, wall_density=wall_density_range) # Pass wall_density_range
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = MazeCTM(grid_size=grid_size, total_ticks=total_ticks, num_neurons=num_neurons, history_len=history_len, sync_window=sync_window, dropout_rate=dropout_rate).to(device) # Pass dropout_rate
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    print(f"Training MazeCTM (grid_size={grid_size}, dropout_rate={dropout_rate})... Performance will be evaluated based on the final prediction after {total_ticks} thought ticks.")
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()

            _, logits_ticks, _ = model(x_batch)

            loss = 0.0
            for t_logits in logits_ticks:
                loss += criterion(t_logits, y_batch)
            loss /= len(logits_ticks)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Average Loss: {total_loss / len(train_loader):.4f}")

    return model, dataset, device # Return model, dataset, and device





In [6]:
# ==========================================
#    Original Call for Training
# ==========================================

# Train and visualize an initial 15x15 maze model with higher wall density
model_10x10, dataset_10x10, device_10x10 = train_maze_ctm_model(
    grid_size=15,
    total_ticks=16,
    wall_density_range=(0.2, 0.4), # Training with a range of wall densities
    num_samples=1500, # Increased number of samples
    num_epochs=300,
    lr=2e-3,
    num_neurons=32, # Default parameters for the baseline model
    history_len=4,
    sync_window=4,
    dropout_rate=0.2 # Applied dropout
)


# The visualize_maze_ctm_reasoning function expects model, dataset, sample_idx, device
viz_anim_output = visualize_maze_ctm_reasoning(model_10x10, dataset_10x10, sample_idx=0, device=device_10x10)

viz_anim_output

Generating synthetic mazes (grid_size=15, wall_density_range=(0.2, 0.4))... Device: cuda
Training MazeCTM (grid_size=15, dropout_rate=0.2)... Performance will be evaluated based on the final prediction after 16 thought ticks.
Epoch 01 | Average Loss: 0.4168
Epoch 05 | Average Loss: 0.3469
Epoch 10 | Average Loss: 0.3138
Epoch 15 | Average Loss: 0.2934
Epoch 20 | Average Loss: 0.2788
Epoch 25 | Average Loss: 0.2633
Epoch 30 | Average Loss: 0.2464
Epoch 35 | Average Loss: 0.2259
Epoch 40 | Average Loss: 0.2103
Epoch 45 | Average Loss: 0.1989
Epoch 50 | Average Loss: 0.1930
Epoch 55 | Average Loss: 0.1811
Epoch 60 | Average Loss: 0.1760
Epoch 65 | Average Loss: 0.1733
Epoch 70 | Average Loss: 0.1713
Epoch 75 | Average Loss: 0.1634
Epoch 80 | Average Loss: 0.1675
Epoch 85 | Average Loss: 0.1613
Epoch 90 | Average Loss: 0.1547
Epoch 95 | Average Loss: 0.1529
Epoch 100 | Average Loss: 0.1541
Epoch 105 | Average Loss: 0.1575
Epoch 110 | Average Loss: 0.1489
Epoch 115 | Average Loss: 0.1550
Ep

### Evaluating Model Performance on an Unseen, In-Distribution Maze

To ensure our trained `MazeCTM` model generalizes well to new, unseen mazes that *do* fall within its training distribution, we'll generate a fresh set of mazes using the same parameters as the training data (e.g., `grid_size=15`, `wall_density` between 0.2 and 0.4).

This test confirms that the model hasn't simply memorized the training samples but has learned the underlying principles of maze solving within its expected operational range.

In [7]:
print('Generating new in-distribution maze dataset...')

# Generate a new dataset with parameters similar to training
in_dist_dataset = SyntheticMazeDataset(
    num_samples=10, # A few samples for visualization
    grid_size=model_10x10.grid_size, # Same grid size as trained model
    wall_density=(0.2, 0.4), # Same wall density range as training
    max_attempts=5000 # Allow sufficient attempts for generation
)

print(f"In-Distribution Dataset generated with {len(in_dist_dataset)} samples.")

# Visualize the model's reasoning on an in-distribution sample
in_dist_viz_anim_output = visualize_maze_ctm_reasoning(
    model=model_10x10, # Use the trained model
    dataset=in_dist_dataset, # Use the new in-distribution dataset
    sample_idx=0, # Visualize the first sample
    device=device_10x10 # Use the device from training
)

in_dist_viz_anim_output

Generating new in-distribution maze dataset...
In-Distribution Dataset generated with 10 samples.


### Evaluating Model Performance on Out-of-Distribution (OOD) Mazes

To test the robustness and generalization capabilities of our trained `MazeCTM` model, it's crucial to evaluate its performance on mazes that differ from the training distribution. For instance, if the model was trained primarily on 15x15 mazes with a 30% wall density, we can create an OOD dataset with a larger grid size (e.g., 20x20) or a significantly different wall density (e.g., 40% or 10%).

This check helps us understand:
*   **Generalization:** How well the model applies its learned patterns to new, unseen problem structures.
*   **Limitations:** Where the model might struggle due to disparities between training and testing environments.

Below, we generate a new `SyntheticMazeDataset` with different parameters (e.g., a larger `grid_size` and slightly different `wall_density`) and then visualize the `MazeCTM`'s reasoning process on a sample from this OOD set.

In [8]:
print('Generating OOD maze dataset (e.g., 15x15 with higher wall density)...')

# Generate an OOD dataset with different parameters
# For example, larger grid size and different wall density
od_dataset = SyntheticMazeDataset(
    num_samples=10, # Only a few samples needed for visualization
    grid_size=model_10x10.grid_size,   # OOD: Use the same grid size as the trained model
    wall_density=0.50, # OOD: Wall density outside the training range (0.2-0.4)
    max_attempts=5000 # Increase attempts for potentially harder generation
)

print(f"OOD Dataset generated with {len(od_dataset)} samples.")

# Visualize the model's reasoning on an OOD sample
# Using the previously trained model and device
ood_viz_anim_output = visualize_maze_ctm_reasoning(
    model=model_10x10, # Use the trained model
    dataset=od_dataset, # Use the OOD dataset
    sample_idx=0,       # Visualize the first sample from the OOD set
    device=device_10x10 # Use the device from training
)

ood_viz_anim_output

Generating OOD maze dataset (e.g., 15x15 with higher wall density)...
OOD Dataset generated with 10 samples.


### Visualizing Model Reasoning with a Custom Start Position

To further understand the `MazeCTM` model's generalization capabilities, we can test its performance on a maze with a starting position that might be less common in the randomly generated training set (e.g., the exact center of the maze). This allows us to observe if the model's learned mechanisms can adapt to new spatial configurations of the start/goal points, even if the maze's `grid_size` and `wall_density` are within its training distribution.

Below, we generate a 15x15 maze (same `grid_size` as trained) with a `wall_density` within the training range (e.g., 0.3). We then manually set the start point to the center of the maze and a new goal point. A ground truth path is recalculated for this specific configuration, and then the `MazeCTM`'s reasoning process is visualized.

In [9]:
def generate_custom_start_maze(grid_size, wall_density, start_coords, goal_coords, max_maze_gen_attempts=100):
    """
    Generates a single maze with specified start and goal coordinates,
    and recalculates the ground truth path, retrying wall generation if necessary.
    """
    H, W = grid_size, grid_size

    for attempt in range(max_maze_gen_attempts):
        # 1. Generate only walls using the logic similar to SyntheticMazeDataset
        walls = np.zeros((H, W), dtype=np.float32)
        # Use a single wall_density if it's a scalar, or sample from range
        if isinstance(wall_density, (tuple, list)): # Handle wall_density_range if passed
            current_wall_density = np.random.uniform(wall_density[0], wall_density[1])
        else:
            current_wall_density = wall_density
        mask = np.random.rand(H, W) < current_wall_density
        walls[mask] = 1.0

        # Ensure start and goal positions are clear of walls
        walls[start_coords[0], start_coords[1]] = 0.0
        walls[goal_coords[0], goal_coords[1]] = 0.0

        # Create new start and goal channels
        start_channel = np.zeros((H, W), dtype=np.float32)
        goal_channel = np.zeros((H, W), dtype=np.float32)
        start_channel[start_coords[0], start_coords[1]] = 1.0
        goal_channel[goal_coords[0], goal_coords[1]] = 1.0

        # 2. Run BFS for the new start/goal to get the correct ground truth path
        path_mask = np.zeros((H, W), dtype=np.float32)
        queue = [(start_coords[0], start_coords[1], [(start_coords[0], start_coords[1])])]
        visited = set([(start_coords[0], start_coords[1])])
        found_path = None

        while queue:
            cy, cx, current_path = queue.pop(0)
            if (cy, cx) == (goal_coords[0], goal_coords[1]):
                found_path = current_path
                break
            for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ny, nx = cy + dy, cx + dx
                if 0 <= ny < H and 0 <= nx < W and walls[ny, nx] == 0.0:
                    if (ny, nx) not in visited:
                        visited.add((ny, nx))
                        queue.append((ny, nx, current_path + [(ny, nx)]))

        if found_path:
            for py, px in found_path:
                path_mask[py, px] = 1.0

            input_grid = np.stack([walls, start_channel, goal_channel], axis=0)
            return torch.tensor(input_grid, dtype=torch.float32), torch.tensor(path_mask, dtype=torch.float32).unsqueeze(0)

    # If no solvable maze found after max_maze_gen_attempts
    print(f"Warning: Could not generate a solvable custom maze after {max_maze_gen_attempts} attempts. Returning an empty maze.")
    # Return an empty maze and mask if no path found after many attempts
    walls = np.zeros((H, W), dtype=np.float32)
    start_channel = np.zeros((H, W), dtype=np.float32)
    goal_channel = np.zeros((H, W), dtype=np.float32)
    start_channel[start_coords[0], start_coords[1]] = 1.0
    goal_channel[goal_coords[0], goal_coords[1]] = 1.0
    input_grid = np.stack([walls, start_channel, goal_channel], axis=0)
    return torch.tensor(input_grid, dtype=torch.float32), torch.zeros((1, H, W), dtype=torch.float32)


class SingleMazeDataset(Dataset):
    """
    A simple Dataset wrapper for a single maze to be used with visualize_maze_ctm_reasoning.
    """
    def __init__(self, maze_input, ground_truth, grid_size):
        self.data = [(maze_input, ground_truth)]
        self.grid_size = grid_size # Ensure grid_size is accessible

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        return self.data[idx]


# Define parameters for the custom maze
grid_size = model_10x10.grid_size # Use the same grid size as the trained model
middle_coords = (grid_size // 2, grid_size // 2)
goal_coords = (0, 0) # Top-left corner as a new goal example
wall_density_for_custom_maze = 0.30 # A density within the training range

print(f"Generating a custom maze with start at {middle_coords} and goal at {goal_coords}...")
custom_input_maze, custom_ground_truth = generate_custom_start_maze(
    grid_size=grid_size,
    wall_density=wall_density_for_custom_maze,
    start_coords=middle_coords,
    goal_coords=goal_coords
)

# Wrap the custom maze into our simple dataset class
custom_dataset = SingleMazeDataset(custom_input_maze, custom_ground_truth, grid_size)

print("Visualizing model's reasoning on the custom-start maze...")
custom_viz_anim_output = visualize_maze_ctm_reasoning(
    model=model_10x10, # Use the previously trained model
    dataset=custom_dataset, # Use the custom dataset
    sample_idx=0,       # Only one sample in the custom dataset
    device=device_10x10 # Use the device from training
)

custom_viz_anim_output

Generating a custom maze with start at (7, 7) and goal at (0, 0)...
Visualizing model's reasoning on the custom-start maze...


## Documentation: How the Maze CTM Model Solves the Task

The `MazeCTM` model is designed to solve mazes by simulating a cognitive process that evolves over internal "thought ticks." It leverages concepts from neural networks, particularly focusing on how synchronized neuronal activity can lead to problem-solving. Here's a breakdown of its mechanism:

1.  **Input Representation:**
    *   The maze is provided as a 3-channel image:
        *   **Channel 0:** Walls (1 for a wall, 0 for an open path).
        *   **Channel 1:** Start position (1 at the start, 0 elsewhere).
        *   **Channel 2:** Goal position (1 at the goal, 0 elsewhere).
    *   This 3D grid input is flattened into a 1D vector (`x_flat`) at the beginning of each reasoning process.

2.  **The Core Loop: Internal Thought Ticks (Temporal Dynamics):**
    *   The model operates for a predefined number of `total_ticks` (e.g., 7 or 16). In each tick `t`, it updates its internal state and refines its understanding of the maze.

3.  **Synaptic Weight Matrix (`W_syn`):**
    *   At each tick `t`, the flattened input `x_flat` is concatenated with the previous tick's aggregate neuronal state (`z_prev`).
    *   This combined vector (`syn_in`) is passed through `W_syn` (a linear layer). This step acts as a dynamic input gate, integrating current maze information with the evolving internal 'memory' (`z_prev`) of the neuronal population. The output `a_t` represents a new input signal to the neuron-level models.

4.  **Neuron-Level Models (`nlms`):**
    *   The `nlms` (NeuronLevelModels) are a set of independent 1D convolutional filters, one for each neuron. They process a short history (`history_len`) of `a_t` signals for each neuron.
    *   Specifically, `a_t` (from `W_syn`) is added to a `pre_hist` buffer. The `nlms` then apply a 1D convolution over this `pre_hist` to generate `z_t`, the current *aggregate state* of each neuron.
    *   `z_t` is a crucial output, representing the learned response of individual neurons based on their recent input history. This `z_t` then becomes `z_prev` for the next tick, creating the recurrent loop.

5.  **Phase Synchrony Engine (`sync_engine`):**
    *   As `z_t` is computed, it's pushed into a `post_win` buffer, which holds the `sync_window` most recent `z_t` states for all neurons.
    *   The `sync_engine` calculates a **synchrony matrix** `S_t` from these recent `z_t` states. `S_t` is a matrix where each entry `S_ij` represents the phase synchrony (e.g., correlation) between neuron `i` and neuron `j`. High synchrony indicates that neurons are firing or fluctuating in a coordinated manner.
    *   From `S_t`, a `s_vec_t` is extracted, which is a vector containing the unique (non-redundant) synchrony values (e.g., the upper triangle of `S_t`). This `s_vec_t` serves as the *current cognitive state* of the model, representing the emergent patterns of neuronal coordination.

6.  **Spatial Output Projection (`W_spatial_out`):**
    *   The `s_vec_t` (the cognitive state derived from neuron synchrony) is then passed through `W_spatial_out` (a small feedforward network).
    *   This network projects the abstract synchrony patterns back into a concrete spatial representation: a grid of logits (`logits_grid`) corresponding to the maze's `grid_size`.
    *   These `logits_grid` values, after applying a sigmoid function, represent the model's confidence that a particular cell is part of the path at that given tick.

7.  **Iterative Refinement:**
    *   This process (`W_syn` -> `nlms` -> `sync_engine` -> `W_spatial_out`) repeats for each `total_ticks`.
    *   Over these ticks, the model's internal states (`z_prev`, `pre_hist`, `post_win`) evolve, and the synchrony patterns (`S_t`, `s_vec_t`) become more structured. This iterative refinement allows the model to "think" and progressively build a more accurate path prediction.

8.  **Final Prediction:**
    *   The `logits_grid` from the *last* thought tick (`total_ticks - 1`) is considered the model's final prediction for the maze path. During training, the loss is often calculated across all ticks to encourage early learning, but the final output is based on the most refined state.

In essence, the `MazeCTM` doesn't just process the maze in one go; it engages in a temporal reasoning process where neurons dynamically synchronize, their collective patterns are interpreted into a spatial context, and this interpretation is refined over several internal computational steps until a solution is reached.

## Mathematical Formulation of the Maze CTM Model

The `MazeCTM` model's dynamics can be described through a set of mathematical equations that govern the neuronal states, synchrony, and spatial output. Let's denote the following:

*   $t$: current thought tick, where $t \in \{1, ..., T\}$ and $T$ is `total_ticks`.
*   $x \in \mathbb{R}^{B \times C \times H \times W}$: Input maze tensor, where $B$ is batch size, $C$ is channels (walls, start, goal), $H$ is height, $W$ is width.
*   $x_{\text{flat}} \in \mathbb{R}^{B \times (C \cdot H \cdot W)}$: Flattened input maze.
*   $z_{t-1} \in \mathbb{R}^{B \times N}$: Aggregate neuronal state from the previous tick, where $N$ is `num_neurons`.
*   $a_t \in \mathbb{R}^{B \times N}$: Synaptic input at tick $t$.
*   $z_t \in \mathbb{R}^{B \times N}$: Aggregate neuronal state at tick $t$.
*   $pre_{\text{hist}} \in \mathbb{R}^{B \times N \times L}$: History of synaptic inputs, where $L$ is `history_len`.
*   $post_{\text{win}} \in \mathbb{R}^{B \times N \times W_{\text{sync}}}$: Window of recent neuronal states, where $W_{\text{sync}}$ is `sync_window`.
*   $S_t \in \mathbb{R}^{B \times N \times N}$: Synchrony matrix at tick $t$.
*   $s_{\text{vec},t} \in \mathbb{R}^{B \times (N(N-1)/2)}$: Vectorized unique elements of the synchrony matrix.
*   $P_t \in \mathbb{R}^{B \times H \times W}$: Output path probability grid at tick $t$.
*   $\sigma(\cdot)$: Sigmoid activation function.
*   $\text{tanh}(\cdot)$: Hyperbolic tangent activation function.

### 1. Synaptic Input Calculation

At each tick $t$, the synaptic input $a_t$ is computed by concatenating the flattened input maze $x_{\text{flat}}$ with the previous neuronal state $z_{t-1}$, and then applying a linear transformation with weights $W_{\text{syn}}$ and a dropout layer:

$$a_t = \text{Dropout}(W_{\text{syn}}([x_{\text{flat}} || z_{t-1}]))$$

Where $[\cdot || \cdot]$ denotes concatenation, and $W_{\text{syn}}$ is a learnable weight matrix.

### 2. Neuron-Level Model (NLMs) Output

The `pre_hist` is updated by shifting and appending $a_t$. The aggregate neuronal state $z_t$ is then derived by applying a 1D convolution (`nlm_conv`) over the `pre_hist` and passing it through a $\text{tanh}$ activation:

$$pre_{\text{hist}} \leftarrow [pre_{\text{hist}}[..., 1:] || a_t]$$
$$z_t = \text{tanh}(\text{nlm_conv}(pre_{\text{hist}}))$$

### 3. Synchrony Engine

The `post_win` is updated by shifting and appending $z_t$. The states in `post_win` are L2-normalized to form $z_{\text{norm}}$, and the synchrony matrix $S_t$ is computed as the outer product (batch matrix multiplication) of $z_{\text{norm}}$ with its transpose:

$$post_{\text{win}} \leftarrow [post_{\text{win}}[..., 1:] || z_t]$$
$$z_{\text{norm}} = \text{normalize}(post_{\text{win}}, p=2, \text{dim}=-1)$$
$$S_t = z_{\text{norm}} z_{\text{norm}}^T$$

From $S_t$, a vectorized representation $s_{\text{vec},t}$ is extracted, typically by taking the unique upper triangular elements (excluding the diagonal).

### 4. Spatial Output Projection

The vectorized synchrony $s_{\text{vec},t}$ is passed through a small feedforward network (`W_spatial_out`) which includes linear layers, ReLU activation, and a dropout layer, to produce logits for each cell in the maze. These logits are then reshaped into a grid format:

$$\text{logits}_{\text{flat},t} = W_{\text{spatial_out}}(s_{\text{vec},t})$$
$$\text{logits}_{\text{grid},t} = \text{reshape}(\text{logits}_{\text{flat},t}, (B, 1, H, W))$$

The final path probabilities $P_t$ for a given tick $t$ are obtained by applying a sigmoid activation to the logits:

$$P_t = \sigma(\text{logits}_{\text{grid},t})$$

### 5. Training Objective

During training, the model typically minimizes the Binary Cross-Entropy (BCE) loss between the predicted path probabilities $P_t$ (or logits) and the ground truth path mask $Y \in \mathbb{R}^{B \times 1 \times H \times W}$. The loss is often summed over all thought ticks to encourage consistent learning throughout the reasoning process:

$$ \mathcal{L} = \frac{1}{T} \sum_{t=1}^{T} \text{BCEWithLogitsLoss}(\text{logits}_{\text{grid},t}, Y) $$

## Applications to Image Processing and Saliency Maps

The principles demonstrated by the `MazeCTM` model can be adapted to various image processing tasks, particularly those involving iterative refinement, spatial reasoning, or contextual understanding.

### Usage in Image Processing Tasks:

1.  **Iterative Image Analysis/Refinement:** The model's concept of `total_ticks` and iterative processing can be applied to tasks where a single pass isn't sufficient:
    *   **Image Segmentation:** Iteratively refine segmentation masks, enhancing boundaries or incorporating global context in each step.
    *   **Image Denoising/Restoration:** Progressively clean noisy images, applying filtering or reconstruction steps over time, leveraging previous iterations.
    *   **Medical Image Analysis:** Iteratively analyze complex medical scans for subtle anomalies, integrating information across different slices or temporal frames.

2.  **Spatial Reasoning and Relational Understanding:**
    *   **Object Relation Recognition:** The `SynchronyEngine` could model relationships between objects or regions in an image, where synchrony represents learned co-occurrence patterns or interactions (e.g., a 'person' with a 'bicycle' on a 'road').
    *   **Contextual Understanding:** For tasks like visual question answering, the model could iteratively process an image and a question, building synchronized representations of relevant image regions and textual concepts to derive an answer.

3.  **Adaptive Feature Extraction:** The `W_syn` layer, which combines raw input with previous neuronal states, acts as an adaptive feature extractor, similar to vision transformers or recurrent convolutional networks that learn to focus on specific features or temporal dynamics.

4.  **Generative Models:** Iterative processes can also be employed in generative models (e.g., GANs, VAEs) to progressively generate images, adding detail with each successive step.

Essentially, any image processing task that benefits from a **sequential, self-correcting, or context-aware approach** can draw inspiration from the `MazeCTM`'s architecture.

### Insights into Saliency Maps:

Yes, the model's inner workings provide direct insights into a form of saliency, particularly through the "Path Confidence" visualization and the `SynchronyEngine`:

1.  **Direct Saliency via "Path Confidence":** The animated heatmaps representing "Path Confidence" (derived from `logits_grids`) serve as spatial saliency maps for the maze-solving task. By observing their evolution, we can discern:
    *   **Focus Areas:** Which maze cells are consistently assigned high confidence as part of the solution path.
    *   **Attention Shift:** How the model's spatial "attention" changes and refines over `total_ticks`.
    *   **Uncertainty:** Low confidence values or broadly distributed probabilities can indicate the model's uncertainty in certain regions.

2.  **Relational Saliency via "Neuron Synchrony S_ij":** While not a direct spatial map, the `SynchronyEngine`'s output (`S_ij` matrices) reveals relational saliency. A high `S_ij` between two neurons suggests a strong correlation or co-activation between the maze features or internal concepts they represent. If these neurons could be mapped back to specific image regions or feature detectors, `S_ij` would highlight which internal components are actively working together to formulate a solution. This provides a window into the model's underlying cognitive dynamics and how different parts of its internal representation are cooperating.